# Write Delta tables to MinIO (S3) with delta-rs

A no-Spark companion to the [GCS example](../gcs): same two layouts, written to a
local MinIO bucket via `deltalake` (delta-rs). `delta-explain` runs on the
**host** and reads the same `s3://` tables.

Runs alongside the GCS notebook -- this one is on port **8889**, GCS on 8888.

Prereqs: `docker compose up -d`, then open http://localhost:8889/?token=delta

In [ ]:
%pip install -q -r requirements.txt

import os
import random

import pyarrow as pa
from deltalake import write_deltalake

# Inside the container MinIO is reachable over the compose network (minio:9000);
# on the host it would be 127.0.0.1:9000. The endpoint comes from the env.
ENDPOINT = os.environ.get("AWS_ENDPOINT_URL", "http://minio:9000")
STORAGE_OPTIONS = {
    "AWS_ENDPOINT_URL": ENDPOINT,
    "AWS_ACCESS_KEY_ID": os.environ.get("AWS_ACCESS_KEY_ID", "minioadmin"),
    "AWS_SECRET_ACCESS_KEY": os.environ.get("AWS_SECRET_ACCESS_KEY", "minioadmin"),
    "AWS_REGION": "us-east-1",
    "AWS_ALLOW_HTTP": "true",
    "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
    "AWS_VIRTUAL_HOSTED_STYLE_REQUEST": "false",
}

random.seed(42)
COUNTRIES = ["DE", "US", "IT"]
N = 6000
rows = [
    {"name": f"user{i}", "age": random.randint(18, 70),
     "country": random.choice(COUNTRIES), "score": round(random.uniform(60, 99), 1)}
    for i in range(N)
]


def table(rs):
    return pa.table({
        "name": pa.array([r["name"] for r in rs]),
        "age": pa.array([r["age"] for r in rs], pa.int32()),
        "country": pa.array([r["country"] for r in rs]),
        "score": pa.array([r["score"] for r in rs]),
    })

print("endpoint:", ENDPOINT, "| rows:", len(rows))

## Step 1 -- a healthy write
Partitioned by `country`, each file a narrow age band, so partition pruning and
data skipping can both rule files out.

In [ ]:
first = True
for c in COUNTRIES:
    crows = sorted([r for r in rows if r["country"] == c], key=lambda r: r["age"])
    chunk = max(1, len(crows) // 4)
    for k in range(0, len(crows), chunk):
        write_deltalake("s3://lake/users", table(crows[k:k + chunk]),
                        partition_by=["country"],
                        mode="overwrite" if first else "append",
                        storage_options=STORAGE_OPTIONS)
        first = False
print("wrote s3://lake/users (partitioned, age-banded)")

### 👉 On the host now
```
delta-explain s3://lake/users \
  --option endpoint=http://localhost:9000 \
  --option allow_http=true \
  --option access_key_id=minioadmin \
  --option secret_access_key=minioadmin \
  --option virtual_hosted_style_request=false \
  --region us-east-1 \
  -w "country = 'DE' AND age > 55" --min-pruning 50
```
Expect strong pruning and **exit 0**. (The host uses `localhost:9000`; the cell
above wrote via `minio:9000` from inside the container.)

## Step 2 -- a careless rewrite (the regression)
No partitioning, no sort -- every file now spans everything, so nothing can be
skipped.

In [ ]:
random.shuffle(rows)
chunk = N // 6
write_deltalake("s3://lake/users-flat", table(rows[:chunk]),
                mode="overwrite", storage_options=STORAGE_OPTIONS)
for k in range(1, 6):
    write_deltalake("s3://lake/users-flat", table(rows[k * chunk:(k + 1) * chunk]),
                    mode="append", storage_options=STORAGE_OPTIONS)
print("wrote s3://lake/users-flat (flat, shuffled)")

### 👉 On the host again -- same command
Pruning collapses toward **0%** and the gate **fails, exit 1**. A notebook just
made every scan read the whole table -- no error, only the gate caught it.